In [ ]:
import pandas as pd
from pathlib import Path
diretorio = Path(r"F:\Estudos\Senai_dados\Semana_15\Dados\Brutos")

Importando bases

In [ ]:
df_2020 = pd.read_csv(fr"{diretorio}\2020.csv", sep=';')
df_2021 = pd.read_csv(fr"{diretorio}\2021.csv", sep=';')
df_2022 = pd.read_csv(fr"{diretorio}\2022.csv", sep=';')
df_2023 = pd.read_csv(fr"{diretorio}\2023.csv", sep=';')
df_2024 = pd.read_csv(fr"{diretorio}\2024.csv", sep=';')
df_2025 = pd.read_csv(fr"{diretorio}\2025.csv", sep=';')
df_2026 = pd.read_csv(fr"{diretorio}\2026.csv", sep=';')

In [ ]:
df_2020.info()

<class 'pandas.DataFrame'>
RangeIndex: 84819 entries, 0 to 84818
Data columns (total 25 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   ano_compra                       84819 non-null  int64  
 1   nome_instituicao                 84819 non-null  str    
 2   esfera                           84819 non-null  str    
 3   cnpj_instituicao                 84819 non-null  str    
 4   municipio_instituicao            84819 non-null  str    
 5   uf                               84819 non-null  str    
 6   compra                           84819 non-null  str    
 7   insercao                         84819 non-null  str    
 8   codigo_br                        84819 non-null  int64  
 9   descricao_catmat                 84819 non-null  str    
 10  unidade_fornecimento             84813 non-null  str    
 11  generico                         37135 non-null  str    
 12  anvisa                       

Verificando nome dos campos

In [ ]:

colunas_por_ano = {}

for ano in range(2020, 2027):
    caminho = diretorio / f"{ano}.csv"
    # nrows=0 lê apenas o cabeçalho, economizando memória e tempo
    # Adicione sep=';' se seus arquivos usarem ponto e vírgula
    df = pd.read_csv(caminho, nrows=0, on_bad_lines='skip')
    colunas_por_ano[ano] = list(df.columns)

# Compara as colunas de cada ano com o primeiro (2020)
primeiro_ano = 2020
cols_base = colunas_por_ano[primeiro_ano]
todas_iguais = True

for ano, cols in colunas_por_ano.items():
    if cols != cols_base:
        print(f"O ano {ano} possui colunas diferentes de 2020.")
        print(f"Colunas de {ano}: {cols}\n")
        todas_iguais = False

if todas_iguais:
    print("Todas as colunas são exatamente iguais em todos os anos!")

Todas as colunas são exatamente iguais em todos os anos!


In [ ]:
import pandas as pd

# Supondo que você já rodou as suas linhas de leitura
df_total = pd.concat([df_2020, df_2021, df_2022, df_2023, df_2024, df_2025, df_2026], ignore_index=True)

print(f"Total de linhas combinadas: {len(df_total)}")

Total de linhas combinadas: 342716


In [ ]:
df_total.head()  # Mostra as primeiras linhas do DataFrame combinado
df_total.info()  # Mostra informações sobre o DataFrame combinado

<class 'pandas.DataFrame'>
RangeIndex: 342716 entries, 0 to 342715
Data columns (total 25 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   ano_compra                       342716 non-null  int64  
 1   nome_instituicao                 342559 non-null  str    
 2   esfera                           342716 non-null  str    
 3   cnpj_instituicao                 342716 non-null  str    
 4   municipio_instituicao            342716 non-null  str    
 5   uf                               342716 non-null  str    
 6   compra                           342716 non-null  str    
 7   insercao                         340588 non-null  str    
 8   codigo_br                        342716 non-null  int64  
 9   descricao_catmat                 342716 non-null  str    
 10  unidade_fornecimento             342691 non-null  str    
 11  generico                         170679 non-null  str    
 12  anvisa       

In [ ]:
df_total['capacidade'].value_counts().head(10)  # Mostra os 10 valores mais comuns na coluna 'capacidade'

capacidade
100.0    16509
10.0     16333
1.0      12423
2.0      11874
20.0      9886
5.0       9185
30.0      6758
120.0     5004
60.0      4655
3.0       3903
Name: count, dtype: int64

### -------- Verificando nulos e duplicadas -------- 

In [ ]:
# Verificando a base total unificada
total_registros = len(df_total)

print(f"Total de registros: {total_registros}")

# Nulos
nulos = df_total.isnull().sum()
if nulos.sum() > 0:
    print("\n--- Nulos por coluna ---")
    nulos_df = pd.DataFrame({
        'Qtd_Nulos': nulos[nulos > 0],
        'Porcentagem (%)': (nulos[nulos > 0] / total_registros) * 100
    })
    print(nulos_df.to_string(formatters={'Porcentagem (%)': '{:.2f}%'.format}))

# Duplicatas
duplicadas = df_total.duplicated().sum()
pct_duplicadas = (duplicadas / total_registros) * 100
print(f"\n--- Duplicatas ---")
print(f"Total de linhas duplicadas: {duplicadas} ({pct_duplicadas:.2f}% do total)")

Total de registros: 342716

--- Nulos por coluna ---
                                 Qtd_Nulos Porcentagem (%)
nome_instituicao                       157           0.05%
insercao                              2128           0.62%
unidade_fornecimento                    25           0.01%
generico                            172037          50.20%
anvisa                              172037          50.20%
capacidade                          218035          63.62%
unidade_medida                      218035          63.62%
unidade_fornecimento_capacidade         25           0.01%

--- Duplicatas ---
Total de linhas duplicadas: 19 (0.01% do total)


Tratando duplicadas

In [ ]:
# 1. Remove as 19 linhas duplicadas
df_total = df_total.drop_duplicates()

Analisando nulos

In [ ]:
# Lista das colunas que possuem poucos nulos e queremos inspecionar
colunas_com_nulos = [
    'nome_instituicao', 
    'insercao', 
]

# Definindo as colunas que você quer enxergar na amostra para dar contexto
# (Trazendo o ano, compra e a descrição do material junto com a coluna nula)
colunas_de_contexto = ['ano_compra', 'compra', 'descricao_catmat']

for coluna in colunas_com_nulos:
    # Filtra apenas as linhas onde a coluna atual é nula
    linhas_vazias = df_total[df_total[coluna].isnull()]
    
    total_vazias = len(linhas_vazias)
    
    if total_vazias > 0:
        print(f"\n--- Amostra de nulos na coluna: {coluna} (Total: {total_vazias}) ---")
        
        # Seleciona as colunas de contexto + a coluna que estamos analisando
        colunas_para_exibir = colunas_de_contexto + [coluna]
        
        # Pega uma amostra de até 5 linhas (ou menos, se houver poucas)
        amostra = linhas_vazias[colunas_para_exibir].head()
        
        # Exibe a amostra
        print(amostra.to_string())
    else:
        print(f"\n--- A coluna {coluna} não tem mais valores nulos ---")


--- Amostra de nulos na coluna: nome_instituicao (Total: 157) ---
        ano_compra      compra                                                                                                  descricao_catmat nome_instituicao
131106        2021  30/06/2021  HIDROCORTISONA, COMPOSIÇÃO:SAL SUCCINATO SÓDICO, CONCENTRAÇÃO:100 MG, FORMA FARMACÊUTICA:PÓ LIÓFILO P/ INJETÁVEL              NaN
131110        2021  30/06/2021                                                DIPIRONA SÓDICA, DOSAGEM:500 MG/ML, APRESENTAÇÃO:SOLUÇÃO INJETÁVEL              NaN
131121        2021  30/06/2021                                       TRAMADOL CLORIDRATO, DOSAGEM:50 MG/ML, FORMA FARMACÊUTICA:SOLUÇÃO INJETÁVEL              NaN
131123        2021  30/06/2021                              FENTANILA, APRESENTAÇÃO:SAL CITRATO, DOSAGEM:0,05 MG/ML, INDICAÇÃO:SOLUÇÃO INJETÁVEL              NaN
131125        2021  30/06/2021                                                                                         FLUO

In [ ]:
# Lista das colunas que possuem poucos nulos e queremos inspecionar
colunas_com_nulos = [
    'unidade_fornecimento', 
    'unidade_fornecimento_capacidade'
]

# Definindo as colunas que você quer enxergar na amostra para dar contexto
# (Trazendo o ano, compra e a descrição do material junto com a coluna nula)
colunas_de_contexto = ['ano_compra', 'compra', 'descricao_catmat']

for coluna in colunas_com_nulos:
    # Filtra apenas as linhas onde a coluna atual é nula
    linhas_vazias = df_total[df_total[coluna].isnull()]
    
    total_vazias = len(linhas_vazias)
    
    if total_vazias > 0:
        print(f"\n--- Amostra de nulos na coluna: {coluna} (Total: {total_vazias}) ---")
        
        # Seleciona as colunas de contexto + a coluna que estamos analisando
        colunas_para_exibir = colunas_de_contexto + [coluna]
        
        # Pega uma amostra de até 5 linhas (ou menos, se houver poucas)
        amostra = linhas_vazias[colunas_para_exibir]
        
        # Exibe a amostra
        print(amostra.to_string())
    else:
        print(f"\n--- A coluna {coluna} não tem mais valores nulos ---")


--- Amostra de nulos na coluna: unidade_fornecimento (Total: 25) ---
        ano_compra      compra                                                                                                                                                                                                                                                                                         descricao_catmat unidade_fornecimento
1247          2020  06/01/2020                                                                                                 CONJUNTO OXIGÊNIO MEDICINAL, ASPECTO FÍSICO:INCOLOR, ODOR:INODORO, GRAU PUREZA:99,60 A 100 PER, TIPO ACONDICIONAMENTO:CILINDRO PORTÁTIL, TOXIDADE:ATÓXICO PEQUENAS QUANTIDADES, APLICAÇÃO:OXIGENOTERAPIA                  NaN
19572         2020  13/03/2020                                                                                                     CONJUNTO OXIGÊNIO MEDICINAL, ASPECTO FÍSICO:INCOLOR, ODOR:INODORO, GRAU PUREZA:99,60 A 100 PER, TIPO 

In [ ]:
# Lista das colunas que possuem poucos nulos e queremos inspecionar
colunas_com_nulos = [
    'generico', 
    'anvisa', 
]

# Definindo as colunas que você quer enxergar na amostra para dar contexto
# (Trazendo o ano, compra e a descrição do material junto com a coluna nula)
colunas_de_contexto = ['ano_compra', 'compra', 'descricao_catmat','unidade_fornecimento']

for coluna in colunas_com_nulos:
    # Filtra apenas as linhas onde a coluna atual é nula
    linhas_vazias = df_total[df_total[coluna].isnull()]
    
    total_vazias = len(linhas_vazias)
    
    if total_vazias > 0:
        print(f"\n--- Amostra de nulos na coluna: {coluna} (Total: {total_vazias}) ---")
        
        # Seleciona as colunas de contexto + a coluna que estamos analisando
        colunas_para_exibir = colunas_de_contexto + [coluna]
        
        # Pega uma amostra de até 5 linhas (ou menos, se houver poucas)
        amostra = linhas_vazias[colunas_para_exibir].head(10)
        
        # Exibe a amostra
        print(amostra.to_string())
    else:
        print(f"\n--- A coluna {coluna} não tem mais valores nulos ---")


--- Amostra de nulos na coluna: generico (Total: 172034) ---
    ano_compra      compra                                                                                                                                                                                                                                                                                                                                                                                                 descricao_catmat unidade_fornecimento generico
1         2020  01/01/2020                                                                                                                                                                                                                                                                                                                  BOLSA VENTILAÇÃO PULMONAR, MATERIAL:BORRACHA, CAPACIDADE:1 L, APLICAÇÃO:VENTILAÇÃO / REINALAÇÃO              UNIDADE      NaN
5         2020  02/01/

In [ ]:
df_total['tipo_compra'].value_counts()  # Mostra os 10 valores mais comuns na coluna 'capacidade'

tipo_compra
ADMINISTRATIVA    337485
JUDICIAL            5212
Name: count, dtype: int64

In [ ]:
pd.crosstab(df_total['unidade_fornecimento'], df_total['generico'])

generico,N,S
unidade_fornecimento,,
AMPOLA,13233,11950
BISNAGA,1933,4267
BLISTER,88,264
BOLSA,1117,635
COMPRIMIDO,33760,51539
CONJUNTO,75,75
CÁPSULA,4334,6163
DOSE,1,0
DRÁGEA,478,377


In [ ]:
df_total.info()

<class 'pandas.DataFrame'>
Index: 342697 entries, 0 to 342715
Data columns (total 25 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   ano_compra                       342697 non-null  int64  
 1   nome_instituicao                 342540 non-null  str    
 2   esfera                           342697 non-null  str    
 3   cnpj_instituicao                 342697 non-null  str    
 4   municipio_instituicao            342697 non-null  str    
 5   uf                               342697 non-null  str    
 6   compra                           342697 non-null  str    
 7   insercao                         340569 non-null  str    
 8   codigo_br                        342697 non-null  int64  
 9   descricao_catmat                 342697 non-null  str    
 10  unidade_fornecimento             342672 non-null  str    
 11  generico                         170663 non-null  str    
 12  anvisa            

In [ ]:
# Lista das colunas que possuem poucos nulos e queremos inspecionar
colunas_com_nulos = [
    'capacidade', 
    'unidade_medida', 
]

# Definindo as colunas que você quer enxergar na amostra para dar contexto
# (Trazendo o ano, compra e a descrição do material junto com a coluna nula)
colunas_de_contexto = ['ano_compra', 'compra', 'descricao_catmat']

for coluna in colunas_com_nulos:
    # Filtra apenas as linhas onde a coluna atual é nula
    linhas_vazias = df_total[df_total[coluna].isnull()]
    
    total_vazias = len(linhas_vazias)
    
    if total_vazias > 0:
        print(f"\n--- Amostra de nulos na coluna: {coluna} (Total: {total_vazias}) ---")
        
        # Seleciona as colunas de contexto + a coluna que estamos analisando
        colunas_para_exibir = colunas_de_contexto + [coluna]
        
        # Pega uma amostra de até 5 linhas (ou menos, se houver poucas)
        amostra = linhas_vazias[colunas_para_exibir].head()
        
        # Exibe a amostra
        print(amostra.to_string())
    else:
        print(f"\n--- A coluna {coluna} não tem mais valores nulos ---")


--- Amostra de nulos na coluna: capacidade (Total: 218027) ---
   ano_compra      compra                                                                                                                                                                                                                  descricao_catmat  capacidade
1        2020  01/01/2020                                                                                                                                   BOLSA VENTILAÇÃO PULMONAR, MATERIAL:BORRACHA, CAPACIDADE:1 L, APLICAÇÃO:VENTILAÇÃO / REINALAÇÃO         NaN
2        2020  01/01/2020                                                                                                                                                                                                         CARVEDILOL, DOSAGEM:25 MG         NaN
3        2020  01/01/2020                                                                                                                       

In [ ]:
#df_total['unidade_fornecimento'].value_counts()  # Mostra os 10 valores mais comuns na coluna 'capacidade'
df_total['unidade_medida'].value_counts()  # Mostra os 10 valores mais comuns na coluna 'capacidade'

unidade_medida
ML       100931
G         13278
UN         5050
DOSES      3288
M          1049
L           616
KG          220
MG          176
MCL          42
MCG           6
CM            5
M3            3
KUI           3
UI            2
DOSE          1
Name: count, dtype: int64

Tratamento de nulos e duplicadas

In [ ]:
# 1. Remover nulos das colunas marginais
df_total = df_total.dropna(subset=['unidade_fornecimento', 'unidade_fornecimento_capacidade'])

# 2. Utilizar 'compra' como proxy para preencher os nulos em 'insercao'
df_total['insercao'] = df_total['insercao'].fillna(df_total['compra'])

# 3. Preencher 'nome_instituicao' utilizando 'cnpj_instituicao'
# Cria um dicionário que mapeia cada CNPJ existente na base ao seu respectivo nome de instituição
mapa_cnpj_nome = df_total.dropna(subset=['nome_instituicao']).drop_duplicates(subset=['cnpj_instituicao']).set_index('cnpj_instituicao')['nome_instituicao'].to_dict()

# Identifica as linhas onde o nome da instituição é nulo
mascara_nulos_nome = df_total['nome_instituicao'].isnull()

# Preenche esses nulos buscando o nome no dicionário através do CNPJ daquela linha
df_total.loc[mascara_nulos_nome, 'nome_instituicao'] = df_total.loc[mascara_nulos_nome, 'cnpj_instituicao'].map(mapa_cnpj_nome)

# Caso algum CNPJ seja exclusivo das linhas nulas e não tenha correspondente na base para mapear, excluímos o resíduo
df_total = df_total.dropna(subset=['nome_instituicao'])

# Verificando o resultado final do tratamento
print("Total de linhas após a limpeza final:", len(df_total))
print("Nulos restantes nas colunas tratadas:")
print(df_total[['nome_instituicao', 'insercao', 'unidade_fornecimento', 'unidade_fornecimento_capacidade']].isnull().sum())

Total de linhas após a limpeza final: 342672
Nulos restantes nas colunas tratadas:
nome_instituicao                   0
insercao                           0
unidade_fornecimento               0
unidade_fornecimento_capacidade    0
dtype: int64


In [ ]:
# Padroniza a coluna generico (garantindo que estão em maiúsculas e sem espaços extras)
df_total['generico'] = df_total['generico'].str.strip().str.upper()

# Preenche nulos da coluna generico com 'NA' (Não Aplicável)
df_total['generico'] = df_total['generico'].fillna('NA')

# Preenche nulos da coluna anvisa com 'SEM_REGISTRO'
df_total['anvisa'] = df_total['anvisa'].fillna('SEM_REGISTRO')
# Verificando o resultado
print("Nulos restantes em capacidade:", df_total['generico'].isnull().sum())
print("Nulos restantes em unidade_medida:", df_total['anvisa'].isnull().sum())

Nulos restantes em capacidade: 0
Nulos restantes em unidade_medida: 0


In [ ]:
import numpy as np

# 1. Definindo o padrão Regex
# Busca números (inteiros ou decimais com vírgula/ponto) seguidos por unidades de medida comuns na saúde/compras
padrao_regex = r'(\d+(?:[.,]\d+)?)\s*(MG|ML|G|L|CM|MM|KG|MCG|UI|MM2|CM2|M|M2)\b'

# 2. Extraindo capacidade e unidade de medida da descrição
# Retorna um DataFrame com duas colunas: 0 (o número) e 1 (a unidade)
extraido = df_total['descricao_catmat'].str.upper().str.extract(padrao_regex)

# 3. Identificando onde os dados originais estão nulos
mascara_nulos = df_total['capacidade'].isnull()

# 4. Preenchendo a 'capacidade' com os números extraídos
# É necessário substituir a vírgula decimal brasileira por ponto antes de converter para float
df_total.loc[mascara_nulos, 'capacidade'] = extraido.loc[mascara_nulos, 0].str.replace(',', '.').astype(float)

# 5. Preenchendo a 'unidade_medida' com o texto extraído
df_total.loc[mascara_nulos, 'unidade_medida'] = extraido.loc[mascara_nulos, 1]

# 6. Tratamento final para os resíduos (itens que realmente não possuem capacidade/medida, como mesas, canetas, etc.)
# Para a capacidade (Numérica): Mantemos como NaN (padrão numérico vazio do pandas) ou preenchemos com 0
df_total['capacidade'] = df_total['capacidade'].fillna(0) 

# Para a unidade (Texto): Preenchemos com 'SEM_MEDIDA' para manter a consistência com o seu 'SEM_REGISTRO'
df_total['unidade_medida'] = df_total['unidade_medida'].fillna('SEM_MEDIDA')

# Verificando o resultado
print("Nulos restantes em capacidade:", df_total['capacidade'].isnull().sum())
print("Nulos restantes em unidade_medida:", df_total['unidade_medida'].isnull().sum())

Nulos restantes em capacidade: 0
Nulos restantes em unidade_medida: 0


In [ ]:
# Verifica como ficou
print("Total de linhas após a limpeza:", len(df_total))
print("Total de nulos restantes:\n", df_total.isnull().sum().sum())

Total de linhas após a limpeza: 342672
Total de nulos restantes:
 0


In [ ]:
df_total.head()

,ano_compra,nome_instituicao,esfera,cnpj_instituicao,municipio_instituicao,uf,compra,insercao,codigo_br,descricao_catmat,...,capacidade,unidade_medida,unidade_fornecimento_capacidade,cnpj_fornecedor,fornecedor,cnpj_fabricante,fabricante,qtd_itens_comprados,preco_unitario,preco_total
0,2020,FUNDO MUNICIPAL DE SAUDE DE MARABA,MUNICIPAL,18.478.187/0001-07,MARABA,PA,01/01/2020,19/01/2024,270019,"GLICONATO DE CÁLCIO, DOSAGEM:10%, APRESENTAÇÃO...",...,10.00,ML,AMPOLA 10.00 ML,04.949.905/0001-63,F CARDOSO E CIA LTDA,01.571.702/0001-98,HALEX ISTAR INDUSTRIA FARMACEUTICA SA,9750,4.500,43875.0
1,2020,FUNDO MUNICIPAL DE SAUDE - MUNICIPIO DE ALTO P...,MUNICIPAL,08.533.932/0001-01,ALTO PARANA,PR,01/01/2020,13/03/2020,243488,"BOLSA VENTILAÇÃO PULMONAR, MATERIAL:BORRACHA, ...",...,1.00,L,UNIDADE,06.974.929/0001-06,NOROESTE MEDICAMENTOS LTDA,73.856.593/0001-66,"PRATI, DONADUZZI & CIA LTDA",3,6.900,20.7
2,2020,MUNICIPIO DE AMERICO BRASILIENSE,MUNICIPAL,43.976.166/0001-50,AMERICO BRASILIENSE,SP,01/01/2020,06/10/2020,267567,"CARVEDILOL, DOSAGEM:25 MG",...,25.00,MG,COMPRIMIDO,01.140.868/0001-50,CIRURGICA OLIMPIO LTDA,61.150.447/0001-31,LABORATORIOS BALDACCI LTDA EM RECUPERACAO JUDI...,36000,0.155,5580.0
3,2020,MUNICIPIO DE AMERICO BRASILIENSE,MUNICIPAL,43.976.166/0001-50,AMERICO BRASILIENSE,SP,01/01/2020,06/10/2020,271356,"ALPRAZOLAM, DOSAGEM:1 MG",...,1.00,MG,COMPRIMIDO,01.140.868/0001-50,CIRURGICA OLIMPIO LTDA,05.254.971/0001-81,ZYDUS NIKKHO FARMACEUTICA LTDA,240000,0.099,23760.0
4,2020,MUNICIPIO DE AMERICO BRASILIENSE,MUNICIPAL,43.976.166/0001-50,AMERICO BRASILIENSE,SP,01/01/2020,06/10/2020,267565,"CARVEDILOL, DOSAGEM:6,25 MG",...,6.25,MG,COMPRIMIDO,01.140.868/0001-50,CIRURGICA OLIMPIO LTDA,61.150.447/0001-31,LABORATORIOS BALDACCI LTDA EM RECUPERACAO JUDI...,36000,0.086,3096.0


In [ ]:
df_total.to_csv(r"F:\Estudos\Senai_dados\Semana_15\Dados\Brutos\base_consolidada_limpa.csv", sep=';', index=False, encoding='utf-8-sig')


# Modelo Estrela

In [ ]:
import pandas as pd

# Supondo que o seu dataframe limpo se chama df_total

# ==========================================
# 1. CRIANDO AS TABELAS DE DIMENSÃO
# ==========================================

# Dimensão Instituição
dim_instituicao = df_total[['cnpj_instituicao', 'nome_instituicao', 'esfera', 'municipio_instituicao', 'uf']].drop_duplicates().reset_index(drop=True)
dim_instituicao.insert(0, 'id_instituicao', dim_instituicao.index + 1)

# Dimensão Fornecedor
dim_fornecedor = df_total[['cnpj_fornecedor', 'fornecedor']].drop_duplicates().reset_index(drop=True)
dim_fornecedor.insert(0, 'id_fornecedor', dim_fornecedor.index + 1)

# Dimensão Fabricante
dim_fabricante = df_total[['cnpj_fabricante', 'fabricante']].drop_duplicates().reset_index(drop=True)
dim_fabricante.insert(0, 'id_fabricante', dim_fabricante.index + 1)

# Dimensão Produto
# Utilizamos o codigo_br em conjunto com a descrição e outras características para garantir unicidade
cols_produto = ['codigo_br', 'descricao_catmat', 'unidade_fornecimento', 'generico', 'anvisa', 'capacidade', 'unidade_medida']
dim_produto = df_total[cols_produto].drop_duplicates().reset_index(drop=True)
dim_produto.insert(0, 'id_produto', dim_produto.index + 1)

# Dimensão Tempo (Calendário)
# Pegamos todas as datas únicas da coluna de compra
datas_unicas = pd.to_datetime(df_total['compra'], format='%Y-%m-%d', errors='coerce').dropna().unique()
dim_tempo = pd.DataFrame({'data_completa': datas_unicas})
dim_tempo['id_tempo'] = dim_tempo['data_completa'].dt.strftime('%Y%m%d').astype(int) # Formato YYYYMMDD como ID
dim_tempo['ano'] = dim_tempo['data_completa'].dt.year
dim_tempo['mes'] = dim_tempo['data_completa'].dt.month
dim_tempo['nome_mes'] = dim_tempo['data_completa'].dt.month_name(locale='pt_BR.utf8') # Opcional: requer locale em pt-br
dim_tempo['dia'] = dim_tempo['data_completa'].dt.day
dim_tempo['trimestre'] = dim_tempo['data_completa'].dt.quarter
dim_tempo['dia_semana'] = dim_tempo['data_completa'].dt.day_name(locale='pt_BR.utf8')

# ==========================================
# 2. CONSTRUINDO A TABELA FATO
# ==========================================

# Fazemos o merge do DataFrame original com as dimensões para recuperar os IDs gerados
fato_compras = df_total.copy()

# Formatando a data da Fato para bater com o id_tempo gerado (YYYYMMDD)
fato_compras['data_compra_dt'] = pd.to_datetime(fato_compras['compra'], format='%Y-%m-%d', errors='coerce')
fato_compras['id_tempo_compra'] = fato_compras['data_compra_dt'].dt.strftime('%Y%m%d').fillna(-1).astype(int)
fato_compras['id_tempo_insercao'] = pd.to_datetime(fato_compras['insercao'], format='%Y-%m-%d', errors='coerce').dt.strftime('%Y%m%d').fillna(-1).astype(int)

# Merge com Instituição
fato_compras = fato_compras.merge(dim_instituicao[['id_instituicao', 'cnpj_instituicao']], on='cnpj_instituicao', how='left')

# Merge com Fornecedor
fato_compras = fato_compras.merge(dim_fornecedor[['id_fornecedor', 'cnpj_fornecedor']], on='cnpj_fornecedor', how='left')

# Merge com Fabricante
fato_compras = fato_compras.merge(dim_fabricante[['id_fabricante', 'cnpj_fabricante']], on='cnpj_fabricante', how='left')

# Merge com Produto
fato_compras = fato_compras.merge(dim_produto[['id_produto', 'codigo_br', 'descricao_catmat']], on=['codigo_br', 'descricao_catmat'], how='left')

# Selecionando apenas os IDs, chaves de dimensão degenerada e as métricas para a Fato final
colunas_fato = [
    'id_tempo_compra', 
    'id_tempo_insercao', 
    'id_instituicao', 
    'id_fornecedor', 
    'id_fabricante', 
    'id_produto', 
    'modalidade_compra', 
    'tipo_compra', 
    'qtd_itens_comprados', 
    'preco_unitario', 
    'preco_total'
]

fato_compras = fato_compras[colunas_fato]

# ==========================================
# 3. EXPORTANDO O MODELO PARA CSV
# ==========================================

dim_instituicao.to_csv('Dim_Instituicao.csv', sep=';', index=False, encoding='utf-8-sig')
dim_fornecedor.to_csv('Dim_Fornecedor.csv', sep=';', index=False, encoding='utf-8-sig')
dim_fabricante.to_csv('Dim_Fabricante.csv', sep=';', index=False, encoding='utf-8-sig')
dim_produto.to_csv('Dim_Produto.csv', sep=';', index=False, encoding='utf-8-sig')
dim_tempo.to_csv('Dim_Tempo.csv', sep=';', index=False, encoding='utf-8-sig')
fato_compras.to_csv('Fato_Compras.csv', sep=';', index=False, encoding='utf-8-sig')